# 01 · Parquet basics — the file format under every lakehouse

**Goal:** understand what's inside a Parquet file, why engines like it, and what it *can't* do on its own — which is exactly the gap Iceberg fills in notebook 02.

**Databricks link:** every Delta or Iceberg table on Databricks is a folder of Parquet files plus metadata. When a query is fast or slow, Parquet is usually why.

**How to use this notebook:** run each cell in order with **Shift + Enter**. After each result, read the **What to notice** note below it. Takes about 30 minutes.

Data: NYC Yellow Taxi trips, January 2024 (≈3 million rows), published by NYC TLC as a Parquet file.

In [ ]:
from lab_helpers import get_spark
from pyspark.sql import functions as F
import pyarrow.parquet as pq
import pandas as pd
import os

spark = get_spark("01-parquet-basics")
RAW = "/home/lab/work/data/raw/yellow_tripdata_2024-01.parquet"
PLAIN = "/home/lab/work/data/plain"       # where this notebook writes its own Parquet
print("Spark", spark.version, "| file size:", round(os.path.getsize(RAW) / 1e6, 1), "MB")

## 1. What's inside a Parquet file

A Parquet file is **columnar**: values of one column are stored together, not row by row.

```
yellow_tripdata_2024-01.parquet
├── Row group 0  (~1M rows)
│   ├── column chunk: VendorID            (compressed)
│   ├── column chunk: tpep_pickup_datetime
│   └── ... one chunk per column
├── Row group 1
├── Row group 2
└── Footer: schema + where every chunk is + (optional) min/max stats per chunk
```

Engines read the **footer first**, then fetch only the column chunks a query needs. We'll read the footer directly with PyArrow — no Spark involved.

In [ ]:
f = pq.ParquetFile(RAW)
meta = f.metadata
print("rows:        ", meta.num_rows)
print("row groups:  ", meta.num_row_groups)
print("columns:     ", meta.num_columns)
print("written by:  ", meta.created_by)
print()
print(f.schema_arrow)

**What to notice**
- The schema (names *and* types) is in the file itself. Nobody has to tell Spark what the columns are.
- ~3M rows are split into 3 **row groups** — the unit an engine can skip.
- `tpep_pickup_datetime` is a `timestamp` **without** a time zone. Spark calls this `timestamp_ntz`.

In [ ]:
# Per-column details for row group 0: compression, sizes, and whether min/max stats exist.
rg = meta.row_group(0)
rows = []
for i in range(rg.num_columns):
    c = rg.column(i)
    rows.append({
        "column": c.path_in_schema,
        "codec": c.compression,
        "compressed_KB": round(c.total_compressed_size / 1024),
        "raw_KB": round(c.total_uncompressed_size / 1024),
        "ratio": round(c.total_uncompressed_size / c.total_compressed_size, 1),
        "has_min_max": c.is_stats_set,
    })
pd.DataFrame(rows)

**What to notice**
- Every column is compressed with **ZSTD**. Repetitive columns (`VendorID`, `payment_type`, `mta_tax`) compress far better than unique ones (timestamps, `trip_distance`).
- `has_min_max` is **False for every column**. The publisher wrote this file *without statistics*. Without min/max per row group, an engine can't skip row groups — it must read them all. Keep this in mind for section 4.

## 2. Reading it with Spark

In [ ]:
df = spark.read.parquet(RAW)
df.printSchema()

In [ ]:
%%time
df.count()

**What to notice**
- `printSchema()` was instant: Spark read only the footer.
- `count()` is fast too: the row count of each row group is in the footer, so Spark doesn't scan the data.

## 3. Columnar reads: only fetch what the query needs

`explain()` shows Spark's plan. Look for two lines in the output:
- **`ReadSchema`** — the columns Spark will actually read from disk.
- **`PushedFilters`** — filters handed to the Parquet reader so it can skip data early.

In [ ]:
expensive = (df.select("tpep_pickup_datetime", "fare_amount")
               .filter(F.col("fare_amount") > 100))
expensive.explain()

**What to notice**
- `ReadSchema` lists only **2 of 19 columns**. The other 17 are never read — that's the columnar win.
- `PushedFilters: [IsNotNull(fare_amount), GreaterThan(fare_amount,100.0)]`. The filter is pushed down, but it can only *skip* row groups if min/max stats exist. This file has none, so every row group is still read and filtered row by row.

## 4. Write your own Parquet — with statistics

Spark writes min/max stats by default. Let's write the same data ourselves and compare.

In [ ]:
(df.repartition(4)                      # 4 output files
   .write.mode("overwrite")
   .parquet(f"{PLAIN}/trips_jan"))

files = sorted(p for p in os.listdir(f"{PLAIN}/trips_jan") if p.endswith(".parquet"))
print(len(files), "files:", files[0], "...")

ours = pq.ParquetFile(f"{PLAIN}/trips_jan/{files[0]}")
col = ours.metadata.row_group(0).column(ours.schema_arrow.names.index("fare_amount"))
print("codec:", col.compression, "| stats set:", col.is_stats_set,
      "| fare_amount min/max in row group 0:", col.statistics.min, "/", col.statistics.max)

**What to notice**
- Spark wrote **Snappy** compression (Spark's default) instead of ZSTD — a setting, not a law.
- Now each row group has **min/max** stats. A query like `fare_amount > 5000` could skip any row group whose max is below 5000 without reading it.
- `_SUCCESS` (in the folder) is how plain Parquet says "the write finished". It's just an empty marker file — readers aren't forced to check it.

## 5. Partitioning: one folder per value

Partitioning splits data into folders by a column's value (Hive style: `pickup_date=2024-01-15/`). A query that filters on that column reads only the matching folders.

In [ ]:
by_day = df.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
by_day.write.mode("overwrite").partitionBy("pickup_date").parquet(f"{PLAIN}/trips_by_day")

folders = sorted(d for d in os.listdir(f"{PLAIN}/trips_by_day") if d.startswith("pickup_date="))
n_files = sum(len([f for f in os.listdir(f"{PLAIN}/trips_by_day/{d}") if f.endswith(".parquet")]) for d in folders)
print(len(folders), "folders,", n_files, "files")
print("first folders:", folders[:4])
print("last folders: ", folders[-3:])

**What to notice**
- You expected 31 folders for January — you got more. The data has trips dated **2002, 2009, Dec 2023 and Feb 2024** (bad clock data from taxi meters). Each odd value becomes its own folder with a tiny file. Real data is messy; partitioning makes the mess visible.
- Many folders × several files each = the **small files problem**: lots of tiny files are slow to list and open, especially on S3.

In [ ]:
part = spark.read.parquet(f"{PLAIN}/trips_by_day")

print("=== filter on the TIMESTAMP column ===")
part.filter("tpep_pickup_datetime >= '2024-01-15' AND tpep_pickup_datetime < '2024-01-16'").explain()

print("=== filter on the PARTITION column ===")
part.filter("pickup_date = '2024-01-15'").explain()

**What to notice** — look for `PartitionFilters` in each plan:
- Filtering on `tpep_pickup_datetime` → `PartitionFilters: []`. Spark reads **every folder**, even though the data is partitioned by day.
- Filtering on `pickup_date` → `PartitionFilters: [... (pickup_date = 2024-01-15)]`. Spark reads **one folder**.

Every user has to know about the extra `pickup_date` column and remember to filter on it. Most won't. Iceberg's **hidden partitioning** (notebook 02) removes this trap.

## 6. What plain Parquet can't do

Three everyday needs that a folder of Parquet files can't handle safely.

### 6a. Delete bad rows
There are trips with negative fares. Let's try to delete them.

In [ ]:
print("negative fares:", df.filter("fare_amount < 0").count())
import re
try:
    spark.sql(f"DELETE FROM parquet.`{PLAIN}/trips_jan` WHERE fare_amount < 0")
except Exception as e:
    reason = next(l for l in str(e).splitlines() if "Exception" in l)
    print("\nDELETE failed:", re.sub(r"Relation \[.*\]", "Relation [...]", reason).strip(": "))

**What to notice:** Spark fails with an internal error — it has no way to delete rows from a plain Parquet folder ("table relation ... parquet"). Parquet files are **immutable** and a folder has no transaction log, so there is no `DELETE`, `UPDATE` or `MERGE`. The only way is: read everything, filter, write a *new* folder, then swap folders — and hope no one reads in between.

### 6b. Change the schema
Suppose the publisher renames `Airport_fee` to `airport_surcharge` starting in February. We append February to the same folder.

In [ ]:
feb = (spark.read.parquet("/home/lab/work/data/raw/yellow_tripdata_2024-02.parquet")
            .withColumnRenamed("Airport_fee", "airport_surcharge"))
feb.write.mode("append").parquet(f"{PLAIN}/trips_jan")

mixed = spark.read.option("mergeSchema", "true").parquet(f"{PLAIN}/trips_jan")
print([c for c in mixed.columns if "irport" in c])   # columns with "airport" in the name
(mixed.groupBy(F.month("tpep_pickup_datetime").alias("month"))
      .agg(F.count("Airport_fee").alias("Airport_fee_values"),
           F.count("airport_surcharge").alias("airport_surcharge_values"))
      .filter("month IN (1, 2)").orderBy("month").show())

**What to notice:** the table now has **two** columns for the same thing. January's values are almost all in `Airport_fee`, February's in `airport_surcharge` (`count()` skips NULLs). The few "crossovers" are trips dated in the other month — each monthly file contains a handful of late or early records. Plain Parquet matches columns **by name**, so a rename looks like "drop one column, add another". Without `mergeSchema`, Spark silently uses one file's schema and hides a column entirely.

> Side note: Spark matches column names **case-insensitively** by default (`spark.sql.caseSensitive=false`), so renaming only the case (`Airport_fee` → `airport_fee`) would *not* split the column. Real renames do.

### 6c. History and safe writes
- `mode("overwrite")` **deletes the old files first**, then writes new ones. If the job dies halfway, the folder is left half-written, and the old data is gone.
- A reader that lists the folder *during* a write can see a mix of old and new files.
- There's no "what did this table look like yesterday?" — no versions, no undo.

## Summary

| Need | Plain Parquet folder | Iceberg table (notebook 02) |
|---|---|---|
| Read only needed columns | ✔ columnar | ✔ (it *is* Parquet underneath) |
| Skip data using min/max | only if the writer added stats | ✔ stats kept in metadata for every file |
| Partition pruning | only if users filter on the extra column | ✔ hidden partitioning: filter on the real column |
| DELETE / UPDATE / MERGE | ✘ rewrite folders by hand | ✔ SQL commands, atomic |
| Rename a column | ✘ looks like drop + add | ✔ columns tracked by ID, metadata-only |
| Safe concurrent reads & writes | ✘ readers can see half a write | ✔ snapshot isolation |
| History / time travel / undo | ✘ | ✔ every write is a snapshot |

## Exercises
1. Using the table from section 1, which column has the **best** compression ratio, and why do you think so?
2. Write `df` partitioned by `payment_type` to `f"{PLAIN}/trips_by_payment"`. How many folders and files do you get? Is this a good partition column? (Hint: think about how queries filter, and how evenly data spreads.)
3. How many January rows have a pickup time **outside** January 2024? Write the query with the DataFrame API.

Next: **`02_iceberg_first_table.ipynb`**.